In [ ]:
#T1
import numpy as np

x = 0.25
m_piston = 40.00 # kg
D = 10.00 # cm
p_atm = 1.00 # bar
p_final = 3.00 # bar
g = 9.81 # m/s^2
h1 = 1.00 # cm
h2 = 4.00 # cm

D = D / 100 # cm to m
A = np.pi * (D/2)**2 # m^2
h1 = h1 / 100 # cm to m
h2 = h2 / 100 # cm to m
p_atm = p_atm * 1e5 # bar to Pa
p_final = p_final * 1e5 # bar to Pa

P1 = p_atm + m_piston * g / A
print (f"初始压强 P1 = {P1*1e-5:.2f} bar")

# 等压膨胀做功
W1 = P1 * A * (h2 - h1)

# 气体内能变化 dU = m*(u2 - u1)
# 计算质量
vf = 1.0528e-3 # m^3/kg
vg = 1.159 # m^3/kg
v1 = vf + x * (vg - vf)
m = A * h1 / v1
print (f"气体质量 m = {m:.2f} kg")

# 计算内能变化
uf = 466.94e03 # J/kg
ug = 2519.7e3 # J/kg
u1 = uf + x * (ug - uf)

V3 = A * h2
v3 = V3 / m
print (f"最终比容 v3 = {v3:.6f} m^3/kg")

# P = 3bar时, vg = 0.6058 m^3/kg, v3>vg, 说明最终状态为过热蒸汽
# P = 3bar时, 440°C v = 1.094 u = 3030.6, 500°C v = 1.187 u = 3130.0
u3 = 3030.6e3 + (v3 - 1.094) * (3130.0e3 - 3030.6e3) / (1.187 - 1.094)

dU = m * (u3 - u1)
print (f"内能变化 dU = {dU:.2f} J")

Q = dU + W1
print (f"热量 Q = {Q:.2f} J")

初始压强 P1 = 1.50 bar
气体质量 m = 0.00 kg
最终比容 v3 = 1.162158 m^3/kg
内能变化 dU = 573.98 J
热量 Q = 609.32 J


In [4]:
#T2 用库方便跑出答案，但是考试是纸质的，哈哈哈哈哈太好了我们有救了
import numpy as np
from CoolProp.CoolProp import PropsSI

# 1. 初始参数输入


p1 = 10.00 # bar
x1 = 0.60
m = 1.92 # kg
p3 = 10.00 # bar
Q23 = 228.00 # kJ

# --- 状态 1 ---
v1 = 1 / PropsSI('D', 'P', p1, 'Q', x1, fluid) # 比体积 m^3/kg
u1 = PropsSI('U', 'P', p1, 'Q', x1, fluid) / 1000 # 比内能 kJ/kg

# --- 状态 2 (过程 1-2: 等容加热到饱和蒸汽 x=1) ---
v2 = v1
x2 = 1.0
u2 = PropsSI('U', 'D', 1/v2, 'Q', x2, fluid) / 1000
p2 = PropsSI('P', 'D', 1/v2, 'Q', x2, fluid)
T2 = PropsSI('T', 'D', 1/v2, 'Q', x2, fluid) # K

# --- 状态 3 (过程 2-3: 等温到 p3=10 bar) ---
T3 = T2
u3 = PropsSI('U', 'P', p3, 'T', T3, fluid) / 1000
v3 = 1 / PropsSI('D', 'P', p3, 'T', T3, fluid)

# --- 各过程计算 (kJ) ---

# 过程 1-2 (等容): W = 0
W12 = 0
Q12 = m * (u2 - u1)

# 过程 2-3 (等温): 已知 Q23
W23 = Q23 - m * (u3 - u2)

# 过程 3-1 (等压): W = P * delta_V
W31 = p3 * m * (v1 - v3) / 1000 # J 转 kJ
Q31 = m * (u1 - u3) + W31

# 循环汇总
W_net = W12 + W23 + W31
Q_net = Q12 + Q23 + Q31

print(f"W_net = {W_net:.2f} kJ")
print(f"Q12  = {Q12:.2f} kJ")
print(f"Q31  = {Q31:.2f} kJ")

ModuleNotFoundError: No module named 'CoolProp'

In [3]:
#T2 手算插值是会死人的啊啊啊啊啊啊
import numpy as np

p1 = 10.00 # bar
x1 = 0.60
m = 1.15 # kg
p3 = 10.00 # bar
Q23 = 228.00 # kJ

p1 = p1*1e05
p3 = p3*1e05

# 过程1-2: 等容加热到饱和蒸汽
# p1 = 10bar , vf = 1.6584e-03, vg = 0.1285, uf = 296.10, ug = 1334.66, T =24.89
vf, vg = 1.6584e-3, 0.1285 # m^3/kg
uf, ug = 296.10, 1334.66 # kJ/kg
v1 = vf + x1 * (vg - vf) # m^3/kg
u1 = uf + x1 * (ug - uf) # kJ/kg
print (f"状态1: v1 = {v1:.6f} m^3/kg, u1 = {u1:.2f} kJ/kg")

# 查氨饱和蒸汽表,
# 16bar, vg=0.0808, Tsat=41.03, ug=1340.97
# 18bar, vg=0.0717, Tsat=45.38, ug=1341.88
p2 = 16e05 + (18e05 - 16e05) * (v1 - 0.0808) / (0.0717 - 0.0808)
T2 = 41.03 + (45.38 - 41.03) * (v1 - 0.0808) / (0.0717 - 0.0808)
u2 = 1340.97 + (1341.88 - 1340.97) * (v1 - 0.0808) / (0.0717 - 0.0808)
print (f"状态2: p2 = {p2/1e05:.2f} bar, T2 = {T2:.2f} °C, u2 = {u2:.2f} kJ/kg")

W_12 = 0 # 等容过程做功为0
Q_12 = m * (u2 - u1)
print (f"过程1-2: Q12 = {Q_12:.2f} kJ")

# 过程2-3: 等温, 已知Q23
# T3 = T2 = 42.28°C, p3  = 10bar, 饱和温度为24.89°C, 说明状态3为过热蒸汽
# 查过热蒸汽表, 10bar, 40°C v=0.13868, u=1369.52
# 10bar, 50°C v=0.14499, u=1391.07
v3 = 0.13868 + (0.14499 - 0.13868) * (T2 - 40) / (50 - 40)
u3 = 1369.52 + (1391.07 - 1369.52) * (T2 - 40) / (50 - 40)
dU = m * (u3 - u2) 
W_23 = Q23 - dU
print (f"过程2-3: W23 = {W_23:.2f} kJ")

# 过程3-1: 等压, W31 = P * delta_V, Q31 = m * (u1 - u3) + W31
W_31 = p3 * m * (v1 - v3) / 1000 # J 转 kJ
Q_31 = m * (u1 - u3) + W_31
print (f"过程3-1: W31 = {W_31:.2f} kJ, Q31 = {Q_31:.2f} kJ")

W_net = W_12 + W_23 + W_31
print (f"循环净做功 W_net = {W_net:.2f} kJ")

状态1: v1 = 0.077763 m^3/kg, u1 = 919.24 kJ/kg
状态2: p2 = 16.67 bar, T2 = 42.48 °C, u2 = 1341.27 kJ/kg
过程1-2: Q12 = 485.34 kJ
过程2-3: W23 = 189.37 kJ
过程3-1: W31 = -71.85 kJ, Q31 = -595.83 kJ
循环净做功 W_net = 117.51 kJ


In [1]:
#T3
#判断的核心依据是压缩因子Z, 如果 Z 非常接近 1（通常偏差在 5% 以内，即 0.95 < Z < 1.05），则认为理想气体模型适用
# Z = PV/(nRT) = P*v/(R*T)
from CoolProp.CoolProp import PropsSI

def check_ideal_gas_robust(fluid, T_val, T_unit, P_val, P_unit):
    # 1. 统一单位换算
    T = (T_val + 459.67) * 5/9 if T_unit == 'F' else T_val + 273.15
    if P_unit == 'psi':
        P = P_val * 6894.757
    elif P_unit == 'bar':
        P = P_val * 1e5
    else:
        P = P_val

    try:
        # 2. 获取临界压力
        pc = PropsSI('PCRIT', fluid) 
        
        # 3. 获取压缩因子
        z = PropsSI('Z', 'T', T, 'P', P, fluid)
        
        # 4. 判定逻辑
        z_check = (0.95 <= z <= 1.05)
        p_check = (P < pc)
        is_ideal = 1 if (z_check and p_check) else 0
        
        return z, P/pc, is_ideal, pc/1e5
    except Exception as e:
        # 如果出错，返回错误信息而不是 None
        return str(e)[:20], 0, "Error", 0

# --- 执行计算 ---
cases = [
    ('Water', 600, 'F', 900, 'psi', "Water 900 psi"),
    ('Water', 600, 'F', 100, 'psi', "Water 100 psi"),
    ('Nitrogen', -20, 'C', 75, 'bar', "Nitrogen 75 bar"),
    ('Nitrogen', -20, 'C', 1, 'bar', "Nitrogen 1 bar")
]

print(f"{'Case':<18} | {'Z':<8} | {'P/Pc':<8} | {'Ideal?'}")
print("-" * 50)

for fluid, t, tu, p, pu, label in cases:
    z, pr, result, pc_val = check_ideal_gas_robust(fluid, t, tu, p, pu)
    
    # 检查 z 是否为数字，避免格式化错误
    z_str = f"{z:<8.4f}" if isinstance(z, float) else f"{z:<8}"
    print(f"{label:<18} | {z_str} | {pr:<8.2f} | {result}")

Case               | Z        | P/Pc     | Ideal?
--------------------------------------------------
Water 900 psi      | 0.8377   | 0.28     | 0
Water 100 psi      | 0.9848   | 0.03     | 1
Nitrogen 75 bar    | 0.9644   | 2.21     | 0
Nitrogen 1 bar     | 0.9993   | 0.03     | 1


In [ ]:
#T4
import numpy as np

# 1. 输入数据
m_air, P_air_i, T_air_i = 2.0, 5.0e5, 350.0  # kg, Pa, K
m_co, P_co_i, T_co_i = 4.0, 2.0e5, 450.0     # kg, Pa, K
k, Ru = 1.395, 8314.0                        # R_u: J/(kmol*K)

# 2. 气体常数与比热计算 (M_air=28.97, M_co=28.01)
R_air = Ru / 28.97
R_co = Ru / 28.01
Cv_air = R_air / (k - 1)
Cv_co = R_co / (k - 1)

# 3. 计算平衡温度 Tf (根据能量守恒: ΔU_air + ΔU_co = 0)
# m1*Cv1*(Tf - T1) + m2*Cv2*(Tf - T2) = 0
Tf = (m_air * Cv_air * T_air_i + m_co * Cv_co * T_co_i) / (m_air * Cv_air + m_co * Cv_co)

# 4. 计算平衡压力 Pf (根据体积守恒: V_total = V_air_i + V_co_i)
V_air_i = (m_air * R_air * T_air_i) / P_air_i
V_co_i = (m_co * R_co * T_co_i) / P_co_i
V_total = V_air_i + V_co_i

# Pf = (m1*R1 + m2*R2) * Tf / V_total
Pf = (m_air * R_air + m_co * R_co) * Tf / V_total

# 5. 计算各自占据的体积
V_air_f = (m_air * R_air * Tf) / Pf
V_co_f = (m_co * R_co * Tf) / Pf

# 输出结果
print(f"(a) 平衡温度 Tf: {Tf:.2f} K")
print(f"(b) 平衡压力 Pf: {Pf/1e5:.4f} bar")
print(f"(c) 空气最终体积: {V_air_f:.4f} m^3")
print(f"    CO 最终体积: {V_co_f:.4f} m^3")

(a) 平衡温度 Tf: 417.41 K
(b) 平衡压力 Pf: 2.3922 bar
(c) 空气最终体积: 1.0015 m^3
    CO 最终体积: 2.0717 m^3


In [ ]:
#T5
from CoolProp.CoolProp import PropsSI
import numpy as np

# 1. 初始参数输入 (统一换算为 SI 单位)
P1 = 3.0e6           # Pa
T1 = 400 + 273.15    # K
AV1 = 85.0 / 60      # m^3/s (从 m^3/min 换算)

P2 = 0.5e6           # Pa
T2 = 180 + 273.15    # K
V2 = 20.0            # m/s (抽汽速度)

P3 = 6.0e3           # Pa (6 kPa)
x3 = 0.83            # 出口干度
mdot_3 = 41000 / 3600 # kg/s (从 kg/h 换算)

fluid = 'Water'

# 2. 状态点物性计算
# 状态 1 (入口)
v1 = 1 / PropsSI('D', 'P', P1, 'T', T1, fluid)
h1 = PropsSI('H', 'P', P1, 'T', T1, fluid)
mdot_1 = AV1 / v1  # 总进气质量流量

# 状态 2 (抽汽)
v2_spec = 1 / PropsSI('D', 'P', P2, 'T', T2, fluid)
h2 = PropsSI('H', 'P', P2, 'T', T2, fluid)

# 状态 3 (出口)
h3 = PropsSI('H', 'P', P3, 'Q', x3, fluid)

# 3. 求解质量流量
mdot_2 = mdot_1 - mdot_3  # 抽汽质量流量

# 4. 计算直径 D2 (AV = mdot * v)
# Area = pi * (D^2 / 4) -> D = sqrt(4 * mdot * v / (pi * V))
Area2 = (mdot_2 * v2_spec) / V2
D2 = np.sqrt(4 * Area2 / np.pi)

# 5. 计算轮机功率 W_dot (单位: W -> kW)
W_dot = mdot_1 * h1 - mdot_2 * h2 - mdot_3 * h3

print(f"1. 抽汽管道直径 D2: {D2:.4f} m")
print(f"2. 轮机输出功率 Power: {W_dot/1000:.2f} kW")

2156053.8653119714
1. 抽汽管道直径 D2: 0.2717 m
2. 轮机输出功率 Power: 13451.96 kW


In [3]:
#T6
import numpy as np

# 初始数据
k = 1.40
T1 = 600.0        # K
P1 = 0.5e6        # Pa
P2 = 0.4e6        # Pa
P3 = 0.3e6        # Pa
R = 0.287         # kJ/(kg*K) 空气气体常数
cv = R / (k - 1)  # kJ/(kg*K)

# --- 状态点计算 (比体积 v) ---
v1 = R * T1 / (P1 / 1e3)
v2 = R * T1 / (P2 / 1e3)  # T2 = T1

# 过程 2-3: P2*v2^k = P3*v3^k (绝热)
v3 = v2 * (P2 / P3)**(1/k)
T3 = T1 * (P3 / P2)**((k-1)/k)

# 状态 4: v4 = v1, P4 = P3 (等压到 v1)
v4 = v1
T4 = T3 * (v4 / v3)

# --- 过程能量计算 (kJ/kg) ---
# 1-2: 等温
w12 = R * T1 * np.log(v2 / v1)
q12 = w12

# 2-3: 绝热
w23 = (P2/1e3 * v2 - P3/1e3 * v3) / (k - 1)
q23 = 0.0

# 3-4: 等压
w34 = (P3 / 1e3) * (v4 - v3)
q34 = cv * (T4 - T3) + w34

# 4-1: 等容
w41 = 0.0
q41 = cv * (T1 - T4)

# --- 循环效率 ---
w_net = w12 + w23 + w34 + w41
q_in = q12 + q41
eta = w_net / q_in

# 输出结果
print(f"Process 1-2: W = {w12:.2f}, Q = {q12:.2f}")
print(f"Process 2-3: W = {w23:.2f}, Q = {q23:.2f}")
print(f"Process 3-4: W = {w34:.2f}, Q = {q34:.2f}")
print(f"Process 4-1: W = {w41:.2f}, Q = {q41:.2f}")
print(f"Thermal Efficiency: {eta:.4f}")

Process 1-2: W = 38.43, Q = 38.43
Process 2-3: W = 33.97, Q = 0.00
Process 3-4: W = -55.29, Q = -193.52
Process 4-1: W = 0.00, Q = 172.20
Thermal Efficiency: 0.0812
